<a href="https://colab.research.google.com/github/Charu1806/RAG_LangChain_Demo/blob/main/Copy_of_rag_visualisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG-Based Company Document Visualisation
### AcmeTech Solutions — LangChain · ChromaDB · UMAP · Plotly · Mistral AI

| Step | What happens |
|---|---|
| 1 | Clone repo & install libraries |
| 2 | Load 6 knowledge base files |
| 3 | Split into 90 chunks |
| 4 | Pre-embedding diagnostics |
| 5 | Generate 384-dim embeddings |
| 6 | Store in ChromaDB |
| 7 | Test retrieval |
| 8 | UMAP 2D & 3D reduction |
| 9 | Build enriched plot DataFrame |
| 10 | Interactive cluster plots |
| 11 | Save PNG + HTML exports |
| 12 | Advanced visualisation |
| 13 | RAG query with Mistral AI ✅ |

---
## Step 1: Clone Repo & Install Libraries

> **Colab users:** Run this cell first. It clones the repo and sets the working
> directory so all file paths resolve correctly.
>
> **Local Jupyter users:** Skip the `git clone` line and set `REPO_DIR` to your
> local project path instead.

In [ ]:
import os, sys

# ── Clone repo (Colab) or point to local path ─────────────────────────────────
REPO_URL = "https://github.com/Charu1806/RAG_LangChain_Demo.git"
REPO_DIR = "RAG_LangChain_Demo"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} ...")
    os.system(f"git clone {REPO_URL}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# ── Install dependencies ──────────────────────────────────────────────────────
os.system("pip install -q -r config/requirements.txt")
print("\n✅ Libraries installed")

Cloning https://github.com/Charu1806/RAG_LangChain_Demo.git ...
Working directory: /content/RAG_LangChain_Demo

✅ Libraries installed


---
## Step 2: Load Documents

Each file in `knowledge_base/` → one `Document` tagged with `category` + `source`.

In [ ]:
#from langchain.schema import Document
from langchain_core.documents import Document
from pathlib import Path

KB_DIR = Path("knowledge_base")

file_category_map = {
    "employee_directory.txt":        "employee",
    "hr_policies.txt":               "hr",
    "finance_tax.txt":               "finance",
    "engineering_documentation.txt": "engineering",
    "customer_support_kb.txt":       "support",
    "product_management.txt":        "product",
}

docs = []
for filename, category in file_category_map.items():
    with open(KB_DIR / filename, "r", encoding="utf-8") as f:
        content = f.read()
    docs.append(Document(
        page_content=content,
        metadata={"category": category, "source": filename}
    ))

print(f"Loaded {len(docs)} files\n")
for doc in docs:
    wc = len(doc.page_content.split())
    print(f"  [{doc.metadata['category']:>12}]  {doc.metadata['source']}  ({wc:,} words)")

doc = docs[0]

print("TYPE:", type(doc))
print("\nMETADATA:", doc.metadata)
print("\nPAGE_CONTENT (first 500 chars):")
print(doc.page_content[:500])
print(f"\n...({len(doc.page_content)} characters total)")

Loaded 6 files

  [    employee]  employee_directory.txt  (5,318 words)
  [          hr]  hr_policies.txt  (5,455 words)
  [     finance]  finance_tax.txt  (5,644 words)
  [ engineering]  engineering_documentation.txt  (5,933 words)
  [     support]  customer_support_kb.txt  (6,527 words)
  [     product]  product_management.txt  (4,217 words)
TYPE: <class 'langchain_core.documents.base.Document'>

METADATA: {'category': 'employee', 'source': 'employee_directory.txt'}

PAGE_CONTENT (first 500 chars):
Document ID: EMP-001
Category: Employee Directory
Title: Employee Profile - Sarah Mitchell

Name: Sarah Mitchell
Employee ID: ACM-1001
Department: Product
Location: San Francisco, CA
Manager: David Okonkwo (VP of Product)
Years of Experience: 8
Skills: Product strategy, roadmap planning, user research, Agile methodology, stakeholder communication, OKR framework, data analysis, wireframing
Current Projects: AcmeTech Platform v4.0 Roadmap, North Star Metrics Initiat

...(39182 characters tot

---
## Step 3: Chunk Documents

Split on `===============================` → **90 chunks, one per document**.

In [ ]:
from collections import Counter

SEPARATOR = "==============================="
chunks = []
for doc in docs:
    for raw in doc.page_content.split(SEPARATOR):
        text = raw.strip()
        if len(text) < 100:
            continue
        chunks.append(Document(page_content=text, metadata=doc.metadata.copy()))

counts = Counter(c.metadata["category"] for c in chunks)
print(f"Total chunks: {len(chunks)}\n")
for cat, n in sorted(counts.items()):
    print(f"  {cat:>12} : {n} chunks")

for i in range(10):
    print(f"\n{'='*80}")
    print(f"CHUNK {i}")
    print(f"Metadata: {chunks[i].metadata}")
    print(chunks[i].page_content)

Total chunks: 90

      employee : 20 chunks
   engineering : 15 chunks
       finance : 15 chunks
            hr : 15 chunks
       product : 10 chunks
       support : 15 chunks

CHUNK 0
Metadata: {'category': 'employee', 'source': 'employee_directory.txt'}
Document ID: EMP-001
Category: Employee Directory
Title: Employee Profile - Sarah Mitchell

Name: Sarah Mitchell
Employee ID: ACM-1001
Department: Product
Location: San Francisco, CA
Manager: David Okonkwo (VP of Product)
Years of Experience: 8
Skills: Product strategy, roadmap planning, user research, Agile methodology, stakeholder communication, OKR framework, data analysis, wireframing
Current Projects: AcmeTech Platform v4.0 Roadmap, North Star Metrics Initiative, Q3 Feature Prioritization
Performance Rating: Exceeds Expectations
Career Goals: Sarah aims to become a Director of Product within the next two years. She is focused on building cross-functional leadership skills and deepening her expertise in AI-driven product devel

In [ ]:
# ============================================================
# STRATEGY 2: Recursive chunking
# ============================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks_recursive = splitter.split_documents(docs)
print("Recursive chunks:", len(chunks_recursive))
for i in range(10):
    print(f"\n{'='*80}")
    print(f"chunks_recursive {i}")
    print(f"Metadata: {chunks[i].metadata}")
    print(chunks[i].page_content)

Recursive chunks: 741

chunks_recursive 0
Metadata: {'category': 'employee', 'source': 'employee_directory.txt'}
Document ID: EMP-001
Category: Employee Directory
Title: Employee Profile - Sarah Mitchell

Name: Sarah Mitchell
Employee ID: ACM-1001
Department: Product
Location: San Francisco, CA
Manager: David Okonkwo (VP of Product)
Years of Experience: 8
Skills: Product strategy, roadmap planning, user research, Agile methodology, stakeholder communication, OKR framework, data analysis, wireframing
Current Projects: AcmeTech Platform v4.0 Roadmap, North Star Metrics Initiative, Q3 Feature Prioritization
Performance Rating: Exceeds Expectations
Career Goals: Sarah aims to become a Director of Product within the next two years. She is focused on building cross-functional leadership skills and deepening her expertise in AI-driven product development. She has expressed strong interest in leading AcmeTech's expansion into enterprise markets and managing a team of senior product managers.



---
## Step 5: Generate Embeddings

`all-MiniLM-L6-v2` — local, CPU-only, no API key needed.  
**Expected shape: `(90, 384)`**

In [ ]:
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

embedding_fn = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

texts = [chunk.page_content for chunk in chunks_recursive]

# LangChain's embed_documents() is the equivalent of .encode() —
# it returns a plain Python list of lists, not a NumPy array
embeddings_list = embedding_fn.embed_documents(texts)
embeddings = np.array(embeddings_list)

print(f"embeddings.shape : {embeddings.shape}")
assert embeddings.shape == (len(chunks_recursive), 384)


#print("✅ Shape check passed: (90, 384)")
# Print first 2 chunks + embeddings
print("\n===== FIRST 2 CHUNKS =====")
for i in range(2):
    print(f"chunks_recursive {i}:")
    print(embeddings[i][:10])


print("\n===== LAST 2 CHUNKS =====")
for i in range(len(chunks_recursive) - 2, len(chunks_recursive)):
    print(f"chunks_recursive {i}:")
    print(embeddings[i][:10])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embeddings.shape : (741, 384)

===== FIRST 2 CHUNKS =====
chunks_recursive 0:
[-0.08611709  0.06768682  0.0196121   0.03508835  0.02586873  0.01332005
  0.01212333  0.0124945  -0.01934885 -0.05960797]
chunks_recursive 1:
[-0.08820415  0.01118222 -0.02756095  0.00107732 -0.03018246  0.06420241
  0.03072275 -0.00249776 -0.02655535  0.02140442]

===== LAST 2 CHUNKS =====
chunks_recursive 739:
[-0.07782121 -0.0689199  -0.00204934 -0.0666861  -0.02109311  0.01776643
  0.04118939  0.01422449  0.03938993 -0.03648139]
chunks_recursive 740:
[-0.082232    0.04699564 -0.04170271 -0.07760753 -0.01753553 -0.02264503
  0.04749214  0.04597744 -0.07037035 -0.04498577]


---
## Step 6: Store in ChromaDB

> ✅ Uses updated `langchain_chroma` and `langchain_huggingface` packages  
> ℹ️ `db.persist()` not needed in chromadb ≥ 0.4 — auto-persists on creation

In [ ]:
import os, shutil, chromadb
from langchain_chroma import Chroma

PERSIST_DIR = "./vector_db"

# Fully release any cached client + delete the old collection on disk
chromadb.api.client.SharedSystemClient.clear_system_cache()
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)
    print("Cleared old vector store")

# Recreate — this time it's guaranteed to be a fresh collection
db = Chroma(
    collection_name="acmetech_docs",
    embedding_function=embedding_fn,
    persist_directory=PERSIST_DIR,
    collection_metadata={"hnsw:space": "cosine"},
)

# Re-add your already-computed embeddings (from the earlier embedding cell)
db._collection.add(
    ids=[f"chunks_recursive{i}" for i in range(len(chunks_recursive))],
    embeddings=embeddings_list,
    documents=texts,
    metadatas=[chunk.metadata for chunk in chunks_recursive],
)

print(f"✅ ChromaDB — {db._collection.count()} documents stored")
print("Collection metadata:", db._collection.metadata)

Cleared old vector store
✅ ChromaDB — 741 documents stored
Collection metadata: {'hnsw:space': 'cosine'}


---
## Step 7: Test Retrieval

| Score | Meaning |
|---|---|
| `< 0.30` |  🔴 Weak match |
| `0.30–0.60` |  🟡 Good match |
| `> 0.60` |   🟢 Strong match |

> Score = cosine distance (lower = more similar)

In [ ]:
print("Chunks in collection:", db._collection.count())
print("Collection metadata:", db._collection.metadata)
def run_query(query, k=3):
    print(f"\nQUERY: {query}")
    for rank, (doc, score) in enumerate(
        db.similarity_search_with_relevance_scores(query, k=k), 1):
        # score is now a normalized 0–1 relevance score: HIGHER = better match
        icon = "🔴" if score < 0.30 else "🟡" if score < 0.60 else "🟢"
        print(f"  #{rank} {icon} [{doc.metadata['category']}] score={score:.4f}")
        print(f"     {doc.page_content[:200].replace(chr(10), ' ')}...")

run_query("How many maternity leave days are allowed?")
run_query("How are Kubernetes deployments configured?")
run_query("How is the annual bonus calculated?")
run_query("How do I reset my account password?")
run_query("What is on the product roadmap?")
run_query("Who manages the engineering team?")

Chunks in collection: 741
Collection metadata: {'hnsw:space': 'cosine'}

QUERY: How many maternity leave days are allowed?
  #1 🟢 [hr] score=0.6703
     Leave Entitlement Eligible employees are entitled to up to 26 weeks of maternity leave. The first 16 weeks are paid at 100% of the employee's base salary. Weeks 17 through 26 are paid at 60% of base s...
  #2 🟢 [hr] score=0.6082
     Leave Entitlement All eligible employees who are the secondary caregiver are entitled to four weeks of fully paid paternity leave, to be taken within six months of the birth or adoption placement date...
  #3 🟡 [hr] score=0.5691
     Notification and Documentation Employees are encouraged to notify their manager and the HR department as early as practicable, and no later than 10 weeks before the expected start of leave. Employees ...

QUERY: How are Kubernetes deployments configured?
  #1 🟢 [engineering] score=0.7469
     Deployment Configuration All service deployments must be defined using Helm charts st

In [ ]:
run_query("How many maternity leave days are allowed?")
run_query("How are Kubernetes deployments configured?")
run_query("How is the annual bonus calculated?")
run_query("How do I reset my account password?")
run_query("What is on the product roadmap?")
run_query("Who manages the engineering team?")


QUERY: How many maternity leave days are allowed?
  #1 🟡 [hr] score=0.5952
     Document ID: HR-003 Category: HR Policies Title: Maternity Leave Policy  AcmeTech Solutions Maternity Leave Policy Effective Date: January 1, 2026 Policy Owner: People Operations  Policy Statement Acm...
  #2 🟡 [hr] score=0.5577
     Document ID: HR-004 Category: HR Policies Title: Paternity Leave Policy  AcmeTech Solutions Paternity Leave Policy Effective Date: January 1, 2026 Policy Owner: People Operations  Policy Statement Acm...
  #3 🟡 [hr] score=0.5140
     Document ID: HR-001 Category: HR Policies Title: Annual Leave Policy  AcmeTech Solutions Annual Leave Policy Effective Date: January 1, 2026 Policy Owner: People Operations  Policy Statement AcmeTech ...

QUERY: How are Kubernetes deployments configured?
  #1 🟢 [engineering] score=0.6806
     Document ID: ENG-001 Category: Engineering Documentation Title: Kubernetes Deployment Guide  AcmeTech Solutions Kubernetes Deployment Guide Version: 3.2 Last

---
## Step 10: Interactive Cluster Plots

| Chart | What to look for |
|---|---|
| 🗺️ 2D clean | Six clearly separated colour islands |
| 🏷️ 2D labelled | Category label at centroid + size = word count |
| 🌐 3D basic | Drag to rotate — depth separates overlapping clusters |
| 🔍 3D sized | Word count sizing in 3D space |

> 💡 Hover = document detail · Click legend = hide/show · Drag (3D) = rotate

In [ ]:
DARK_BG   = "#0f0f1a"
LEGEND_BG = "rgba(255,255,255,0.08)"
LEGEND_BC = "rgba(255,255,255,0.20)"

def dark_layout(fig, is_3d=False):
    axis = dict(showgrid=False, zeroline=False, showticklabels=False, title="")
    base = dict(paper_bgcolor=DARK_BG, font_color="white", title_font_size=19,
                legend=dict(title="Category", bgcolor=LEGEND_BG,
                            bordercolor=LEGEND_BC, borderwidth=1))
    if is_3d:
        base["scene"] = dict(bgcolor=DARK_BG,
            xaxis={**axis, "backgroundcolor": DARK_BG},
            yaxis={**axis, "backgroundcolor": DARK_BG},
            zaxis={**axis, "backgroundcolor": DARK_BG})
    else:
        base["plot_bgcolor"] = DARK_BG
        base["xaxis"] = axis
        base["yaxis"] = axis
    return fig.update_layout(**base)

In [ ]:
fig1 = px.scatter(df, x="x", y="y", color="category",
    color_discrete_map=CATEGORY_COLORS, hover_name="title",
    hover_data={"doc_id": True, "word_count": True, "preview": True,
                "x": False, "y": False, "category": False},
    title="🗺️  Vector Embedding Clusters — UMAP 2D  (384 → 2 dimensions)",
    width=1050, height=680)
fig1.update_traces(marker=dict(size=11, opacity=0.88, line=dict(width=0.6, color="white")))
dark_layout(fig1).show()

In [ ]:
fig2 = px.scatter(df, x="x", y="y", color="category",
    color_discrete_map=CATEGORY_COLORS, size="word_count", size_max=22,
    hover_name="title",
    hover_data={"doc_id": True, "word_count": True, "preview": True,
                "x": False, "y": False, "category": False},
    title="🏷️  UMAP 2D — Category Labels + Point Size = Word Count",
    width=1050, height=680)
fig2.update_traces(marker=dict(opacity=0.82, line=dict(width=0.5, color="white")))
for cat, color in CATEGORY_COLORS.items():
    sub = df[df["category"] == cat]
    fig2.add_annotation(x=sub["x"].mean(), y=sub["y"].mean(),
        text=f"<b>{cat.upper()}</b>", showarrow=False,
        font=dict(size=14, color=color), bgcolor="rgba(15,15,26,0.70)", borderpad=5)
dark_layout(fig2).show()

In [ ]:
fig3 = px.scatter_3d(df, x="x3", y="y3", z="z3", color="category",
    color_discrete_map=CATEGORY_COLORS, hover_name="title",
    hover_data={"doc_id": True, "word_count": True, "preview": True,
                "x3": False, "y3": False, "z3": False, "category": False},
    title="🌐  Vector Embedding Clusters — UMAP 3D  (384 → 3 dimensions)",
    width=1050, height=780)
fig3.update_traces(marker=dict(size=6, opacity=0.90, line=dict(width=0.3, color="white")))
dark_layout(fig3, is_3d=True)
fig3.update_layout(scene_camera=dict(eye=dict(x=1.5, y=1.5, z=0.8)))
fig3.show()

In [ ]:
fig4 = px.scatter_3d(df, x="x3", y="y3", z="z3", color="category",
    color_discrete_map=CATEGORY_COLORS, size="word_count", size_max=14,
    hover_name="title",
    hover_data={"doc_id": True, "word_count": True, "category": True,
                "preview": True, "x3": False, "y3": False, "z3": False},
    title="🔍  UMAP 3D — Point Size = Word Count  |  Hover = Document Detail",
    width=1050, height=780)
fig4.update_traces(marker=dict(opacity=0.85, line=dict(width=0.2, color="white")))
dark_layout(fig4, is_3d=True)
fig4.update_layout(scene_camera=dict(eye=dict(x=1.8, y=0.8, z=0.6)))
fig4.show()

---
## Step 12: Advanced Visualisation

| Chart | What it shows | Why it's powerful |
|---|---|---|
| **A — Full Text Hover** | Complete document on hover | See exactly what the DB contains |
| **B — Category Filter** | Dropdown to isolate one cluster | Best for live demos |
| **C — Similarity Spotlight** | Query ⭐ + lines to top-5 retrieved | Makes retrieval *spatial and visible* |

In [ ]:
import textwrap

def wrap_text(text, width=80, max_lines=20):
    lines = []
    for para in text.strip().splitlines():
        para = para.strip()
        if not para:
            continue
        lines.extend(textwrap.wrap(para, width=width))
        if len(lines) >= max_lines:
            break
    clipped = lines[:max_lines]
    if len(lines) > max_lines:
        clipped.append("… [truncated]")
    return "<br>".join(clipped)

df["full_text_wrapped"] = [wrap_text(c.page_content) for c in chunks]

figA = px.scatter(df, x="x", y="y", color="category",
    color_discrete_map=CATEGORY_COLORS, hover_name="title",
    hover_data={"doc_id": True, "category": True, "word_count": True,
                "full_text_wrapped": True, "x": False, "y": False},
    title="📄  Chart A — Full Document Text on Hover", width=1100, height=720)
figA.update_traces(
    marker=dict(size=12, opacity=0.88, line=dict(width=0.6, color="white")),
    hoverlabel=dict(bgcolor="#1a1a2e", font_size=12, namelength=-1))
for cat, color in CATEGORY_COLORS.items():
    sub = df[df["category"] == cat]
    figA.add_annotation(x=sub["x"].mean(), y=sub["y"].mean(),
        text=f"<b>{cat.upper()}</b>", showarrow=False,
        font=dict(size=13, color=color), bgcolor="rgba(15,15,26,0.70)", borderpad=4)
dark_layout(figA).show()

In [ ]:
figB = go.Figure()
categories = sorted(df["category"].unique())

for cat in categories:
    sub = df[df["category"] == cat]
    figB.add_trace(go.Scatter(
        x=sub["x"], y=sub["y"], mode="markers", name=cat,
        marker=dict(size=11, color=CATEGORY_COLORS[cat],
                    opacity=0.88, line=dict(width=0.6, color="white")),
        hovertemplate=("<b>%{customdata[0]}</b><br>ID: %{customdata[1]}<br>"
                       "Words: %{customdata[2]}<br><i>%{customdata[3]}</i><extra></extra>"),
        customdata=sub[["title", "doc_id", "word_count", "preview"]].values,
    ))

buttons = [dict(label="All Categories", method="update",
                args=[{"visible": [True]*len(categories)},
                      {"title": "🏷️  All Categories"}])]
for i, cat in enumerate(categories):
    buttons.append(dict(label=cat.capitalize(), method="update",
        args=[{"visible": [j == i for j in range(len(categories))]},
              {"title": f"🔍  {cat.upper()} only"}]))

figB.update_layout(
    title="🏷️  Chart B — Category Filter Dropdown", title_font_size=19,
    updatemenus=[dict(buttons=buttons, direction="down", showactive=True,
        x=0.01, xanchor="left", y=1.12, yanchor="top",
        bgcolor="#1e1e3a", bordercolor="rgba(255,255,255,0.3)",
        font=dict(color="white", size=13))],
    plot_bgcolor=DARK_BG, paper_bgcolor=DARK_BG, font_color="white",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    legend=dict(title="Category", bgcolor=LEGEND_BG, bordercolor=LEGEND_BC, borderwidth=1),
    width=1100, height=720)
figB.show()

In [ ]:
SPOTLIGHT_QUERY = "How many days of maternity leave are employees entitled to?"
TOP_K = 5

query_vec  = embedding_model.encode([SPOTLIGHT_QUERY])[0]
normed_emb = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-10)
query_norm = query_vec  / (np.linalg.norm(query_vec) + 1e-10)
sims       = normed_emb @ query_norm
top_k_idx  = np.argsort(sims)[::-1][:TOP_K]
qx = df.iloc[top_k_idx]["x"].mean()
qy = df.iloc[top_k_idx]["y"].mean()

figC = go.Figure()
figC.add_trace(go.Scatter(x=df["x"], y=df["y"], mode="markers", name="All documents",
    marker=dict(size=8, color=[CATEGORY_COLORS[c] for c in df["category"]], opacity=0.20),
    hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<extra></extra>",
    customdata=df[["title", "preview"]].values))

top_df = df.iloc[top_k_idx].copy()
top_df["rank"] = range(1, TOP_K + 1)
top_df["sim"]  = sims[top_k_idx].round(4)
for _, row in top_df.iterrows():
    figC.add_shape(type="line", x0=qx, y0=qy, x1=row["x"], y1=row["y"],
        line=dict(color="rgba(255,255,255,0.35)", width=1.5, dash="dot"))

figC.add_trace(go.Scatter(x=top_df["x"], y=top_df["y"], mode="markers+text",
    name=f"Top {TOP_K} retrieved",
    marker=dict(size=16, color=[CATEGORY_COLORS[c] for c in top_df["category"]],
                opacity=1.0, line=dict(width=2, color="white")),
    text=[f"#{r}" for r in top_df["rank"]], textposition="top center",
    textfont=dict(size=11, color="white"),
    hovertemplate=("<b>#%{customdata[0]} — %{customdata[1]}</b><br>"
                   "Category: %{customdata[2]}<br>Similarity: %{customdata[3]}<br>"
                   "<i>%{customdata[4]}</i><extra></extra>"),
    customdata=top_df[["rank", "title", "category", "sim", "preview"]].values))

figC.add_trace(go.Scatter(x=[qx], y=[qy], mode="markers+text", name="Query",
    marker=dict(size=20, color="#FFD700", symbol="star", line=dict(width=2, color="white")),
    text=["QUERY"], textposition="bottom center", textfont=dict(size=12, color="#FFD700"),
    hovertemplate=f"<b>Query</b><br>{SPOTLIGHT_QUERY}<extra></extra>"))

figC.update_layout(
    title=f'🔍  Chart C — Similarity Spotlight: "{SPOTLIGHT_QUERY[:55]}..."',
    title_font_size=17, plot_bgcolor=DARK_BG, paper_bgcolor=DARK_BG, font_color="white",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title=""),
    legend=dict(bgcolor=LEGEND_BG, bordercolor=LEGEND_BC, borderwidth=1),
    width=1100, height=740)
figC.show()

print(f"\nTop {TOP_K} nearest docs for: \"{SPOTLIGHT_QUERY}\"\n")
for rank, idx in enumerate(top_k_idx, 1):
    row = df.iloc[idx]
    print(f"  #{rank}  [{row['category']:>12}]  {row['doc_id']}  sim={sims[idx]:.4f}  {row['title']}")

---
## Step 13: RAG Query with Mistral AI

```
User question → ChromaDB (top-4 chunks) → Mistral AI → Grounded answer + sources
```

**Get a free Mistral API key:** https://console.mistral.ai → API Keys → Create new key

**Set it in Colab:**  
Left panel → 🔑 Secrets → add `MISTRAL_API_KEY`

| Setting | Value | Why |
|---|---|---|
| Model | `mistral-small-latest` | Fast, accurate, no daily cap on free tier |
| `temperature` | `0.2` | Low = factual, consistent |
| `k` (retrieval) | `4` | Top 4 chunks as context |
| Prompt | Strict context-only | Prevents hallucination |

In [ ]:
import os, time

# ── Load key from Colab Secrets ────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    pass  # running locally — set via: export MISTRAL_API_KEY='your-key'

MISTRAL_API_KEY= os.environ.get("MISTRAL_API_KEY", "")
if not MISTRAL_API_KEY:
    raise EnvironmentError(
        "MISTRAL_API_KEY not set.\n"
        "Get a free key at: https://console.mistral.ai"
    )
print(f"Key ends with: ...{MISTRAL_API_KEY[-4:]}")

✅ API key loaded from Colab Secrets
Key ends with: ...xO70


In [ ]:
!pip install -q langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
    max_output_tokens=1024,
)

SYSTEM_PROMPT = """You are a helpful assistant for AcmeTech Solutions employees.
Answer the user's question using ONLY the context documents provided below.

Rules:
- Be concise and specific — answer the question directly.
- If the answer is a number, date, or policy detail, state it clearly.
- If the context does not contain enough information, say:
  "I couldn't find a specific answer in the AcmeTech knowledge base."
- Never make up information not present in the context.
- At the end of your answer, list the document IDs you used as sources.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
])

retriever     = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})
combine_chain = create_stuff_documents_chain(llm, prompt)
rag_chain     = create_retrieval_chain(retriever, combine_chain)

print("✅ RAG chain ready  (ChromaDB → gemini-2.5-flash)")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 10.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 0.3.1 requires langchain-core<1.0.0,>=0.3.70, but you have langchain-core 1.6.1 which is incompatible.
langchain 0.3.30 requires langchain-core<1.0.0,>=0.3.85, but you have langchain-core 1.6.1 which is incompatible.


ImportError: cannot import name 'ContextOverflowError' from 'langchain_core.exceptions' (/usr/local/lib/python3.13/dist-packages/langchain_core/exceptions.py)

In [ ]:
!pip install -q -U langchain langchain-core langchain-community langchain-mistralai langchain-huggingface langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 117.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 306.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 2.2 MB/s eta 0:00:00


ChatMistralAI — the LLM wrapper (same role as ChatGoogleGenerativeAI from earlier — a LangChain-standard interface around a specific provider's chat model)
create_retrieval_chain — a LangChain helper function that builds the whole RAG pipeline: "take a question → retrieve relevant chunks → hand them to the LLM → return an answer," all in one object
create_stuff_documents_chain — builds the piece that "stuffs" (inserts) retrieved chunks into your prompt before sending it to the LLM
ChatPromptTemplate — a reusable, fill-in-the-blank prompt structure

In [ ]:
from langchain_mistralai import ChatMistralAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=MISTRAL_API_KEY,
    temperature=0.2,
    max_tokens=1024,
)

SYSTEM_PROMPT = """You are a helpful assistant for AcmeTech Solutions employees.
Answer the user's question using ONLY the context documents provided below.

Rules:
- Be concise and specific — answer the question directly.
- If the answer is a number, date, or policy detail, state it clearly.
- If the context does not contain enough information, say:
  "I couldn't find a specific answer in the AcmeTech knowledge base."
- Never make up information not present in the context.
- At the end of your answer, list the document IDs you used as sources.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
])

retriever     = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})
combine_chain = create_stuff_documents_chain(llm, prompt)
rag_chain     = create_retrieval_chain(retriever, combine_chain)

print("✅ RAG chain ready  (ChromaDB → mistral-small-latest)")

✅ RAG chain ready  (ChromaDB → mistral-small-latest)


In [ ]:
import textwrap

RATE_LIMIT_PAUSE = 1.5   # Mistral free tier: 1 req/sec
_last_call_time  = 0.0

def ask(question: str) -> None:
    global _last_call_time
    elapsed = time.time() - _last_call_time
    if elapsed < RATE_LIMIT_PAUSE and _last_call_time > 0:
        time.sleep(RATE_LIMIT_PAUSE - elapsed)

    print("=" * 70)
    print(f"  ❓  {question}")
    print("=" * 70)

    _last_call_time = time.time()
    response = rag_chain.invoke({"input": question})
    answer   = response["answer"]
    sources  = response["context"]

    print("\n  💬  Answer:\n")
    for line in textwrap.fill(answer, width=66, subsequent_indent="  ").splitlines():
        print(f"  {line}")

    print(f"\n  📄  Sources ({len(sources)} chunks retrieved):\n")
    seen = set()
    for doc in sources:
        cat, source = doc.metadata.get("category","?"), doc.metadata.get("source","?")
        if (cat, source) not in seen:
            seen.add((cat, source))
            snippet = doc.page_content.strip()[:80].replace("\n", " ")
            print(f"    [{cat:>12}]  {source}")
            print(f"               \"{snippet}...\"")
    print()

In [ ]:
ask("How many weeks of maternity leave does AcmeTech provide, and what is the pay structure?")
ask("What are the rules for carrying over unused annual leave?")

  ❓  How many weeks of maternity leave does AcmeTech provide, and what is the pay structure?

  💬  Answer:

  AcmeTech provides up to **26 weeks** of maternity leave with the
    following pay structure: - **First 16 weeks**: 100% of base
    salary. - **Weeks 17–26**: 60% of base salary.  *Document IDs:
    Leave Entitlement, AcmeTech Solutions Maternity Leave Policy*

  📄  Sources (4 chunks retrieved):

    [          hr]  hr_policies.txt
               "Leave Entitlement Eligible employees are entitled to up to 26 weeks of maternity..."

  ❓  What are the rules for carrying over unused annual leave?

  💬  Answer:

  - A maximum of five unused annual leave days may be carried over
    from one calendar year to the next. - Any accrued leave in
    excess of five days at year-end will be forfeited unless an
    employee has an active approved leave request that could not be
    fulfilled due to business requirements. - Employees must request
    carryover extensions in writing to their

In [ ]:
ask("How is the annual performance bonus calculated for a mid-level employee?")
ask("What is the procurement approval limit for a Director-level employee?")

  ❓  How is the annual performance bonus calculated for a mid-level employee?

  💬  Answer:

  The annual performance bonus for a **mid-level employee (Level
    4–5)** is calculated as follows:  **Target Bonus**: 10% of
    annual base salary.  **Actual Bonus Payout**: = **Target Bonus**
    × **Company Performance Multiplier** × **Individual Performance
    Multiplier**  - **Company Performance Multiplier**: Set by the
    executive team and board (ranges from 0 to 1.5). - **Individual
    Performance Multiplier**:   - Outstanding: 1.25   - Exceeds
    Expectations: 1.10   - Meets Expectations: 1.0   - Needs
    Improvement: 0.5   - Unsatisfactory: 0  **Payout Timing**: Paid
    in the February payroll cycle of the year following the
    performance year.  **Sources**: FIN-005

  📄  Sources (4 chunks retrieved):

    [     finance]  finance_tax.txt
               "Document ID: FIN-005 Category: Finance & Tax Title: Bonus Calculation Framework ..."
    [          hr]  hr_policies.txt


In [ ]:
ask("What health checks must every Kubernetes service implement before going to production?")
ask("What is the incident severity classification for a complete platform outage?")

  ❓  What health checks must every Kubernetes service implement before going to production?

  💬  Answer:

  Every Kubernetes service must implement **liveness and readiness
    probes** before going to production.  - **Readiness probes**
    must verify that the service can handle traffic, including
    validating database connectivity and downstream service
    availability. - **Liveness probes** must check that the process
    has not entered an unrecoverable state. - Probe failure
    thresholds and timeouts must be tuned to reflect the service's
    actual startup and response characteristics.  Services without
    configured probes will be flagged during deployment validation.
    **Document IDs used:** ENG-001

  📄  Sources (4 chunks retrieved):

    [ engineering]  engineering_documentation.txt
               "Document ID: ENG-001 Category: Engineering Documentation Title: Kubernetes Deplo..."
    [     product]  product_management.txt
               "Document ID: PM-007 Catego

In [ ]:
ask("What should a customer do if they do not receive a password reset email?")
ask("How do I configure Single Sign-On using SAML 2.0 in AcmeTech?")

In [ ]:
ask("What are AcmeTech's three strategic product themes for FY2026?")
ask("How does AcmeTech calculate an OKR score and what does 0.7 mean?")

In [ ]:
ask("What are Sarah Mitchell's current projects and performance rating?")
ask("Who is the Engineering Director and how many engineers do they manage?")

In [ ]:
# ── Try your own question ─────────────────────────────────────────────────────
ask("Your question here")